# 02 · LoRA from scratch

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/post-training/02_lora.ipynb) &nbsp; 📖 Concept deck: [LoRA and QLoRA](https://nilesh-salpe.github.io/visual-ai/post-training/03-lora-qlora/) &nbsp; ⏱️ Suggested time: **40 min**

**You will build:** a LoRA linear layer, weight merging for zero-latency inference, parameter counting for real models, and a training loop that shows when a low rank is enough.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def log_softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    return x - np.log(np.exp(x).sum(axis=axis, keepdims=True))

def log_sigmoid(z):
    return -np.logaddexp(0, -z)

## Task 1: The LoRA layer

Freeze the pretrained weight $W_0$ ([d_in, d_out]) and learn a low-rank update $\Delta W = AB$ with $A \in \mathbb{R}^{d_{in}\times r}$, $B \in \mathbb{R}^{r\times d_{out}}$:
$$y = xW_0 + \tfrac{\alpha}{r}\, xAB$$
Initialization (LoRA paper): *"a random Gaussian initialization for A and zero for B, so ΔW = BA is zero at the beginning of training"*. The model starts exactly at $W_0$. (The paper writes $BA$ for column vectors; we use row vectors, so it's $AB$.)

Implement `__init__` (A ~ N(0, 1/d_in), B = 0), `forward(x)` and `merged()` (returns $W_0 + \frac{\alpha}{r}AB$).

In [ ]:
class LoRALinear:
    def __init__(self, W0, r, alpha, seed=0):
        self.W0, self.r, self.alpha = W0, r, alpha
        rng = np.random.default_rng(seed)
        # TODO: self.A ~ normal(0, 1/sqrt(d_in)) shape [d_in, r]; self.B zeros [r, d_out]
        raise NotImplementedError

    def forward(self, x):
        raise NotImplementedError

    def merged(self):
        raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_LoRALinear(cls):
    r_ = np.random.default_rng(1); W0 = r_.normal(size=(12, 7)); x = r_.normal(size=(5, 12))
    m = cls(W0, r=3, alpha=6)
    assert m.A.shape == (12, 3) and m.B.shape == (3, 7), "A is [d_in, r], B is [r, d_out]"
    assert np.allclose(m.forward(x), x @ W0), "with B = 0 the layer must equal the frozen W0"
    m.B = r_.normal(size=(3, 7))
    assert np.allclose(m.forward(x), x @ W0 + 2.0 * x @ m.A @ m.B), "scale alpha/r = 2 here"
    assert np.allclose(x @ m.merged(), m.forward(x)), "merged weight must give identical outputs"

check("LoRALinear", _test_LoRALinear, LoRALinear)

## Task 2: Count trainable parameters

One adapted matrix of shape [d_in, d_out] adds $r(d_{in} + d_{out})$ trainable parameters.
`targets` is a list of `(d_in, d_out)` shapes adapted in **each layer**; return the total over `n_layers`.

The LoRA paper mostly adapts only $W_q$ and $W_v$. For Llama 3 8B ($W_q$: 4096×4096, $W_v$: 4096×1024 with GQA, 32 layers) at r = 8 that is **3,407,872**, about 0.04% of 8.03B.

In [ ]:
def lora_param_count(targets, n_layers, r):
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_lora_param_count(f):
    assert f([(4096, 4096), (4096, 1024)], 32, 8) == 3_407_872, f"got {f([(4096, 4096), (4096, 1024)], 32, 8):,}"
    assert f([(10, 20)], 1, 1) == 30
    assert f([(10, 20)], 3, 4) == 360

check("lora_param_count", _test_lora_param_count, lora_param_count)

## Task 3: Train only A and B

Fit a frozen $W_0$ plus LoRA to targets $Y = XW^\star$, where the true update $W^\star - W_0$ has rank 4. Minimize $\frac{1}{N}\|XW_0 + sXAB - Y\|^2$ (with $s = \alpha/r$) by gradient descent on **A and B only**:
$$E = XW_0 + sXAB - Y,\quad \nabla_B = \tfrac{2s}{N}(XA)^\top E,\quad \nabla_A = \tfrac{2s}{N}X^\top (E B^\top)$$
Return the trained layer and the loss history. The tests check that rank ≥ 4 fits almost perfectly and rank 1 cannot.

In [ ]:
def train_lora(X, Y, W0, r, alpha=None, steps=2000, lr=0.05, layer_cls=None):
    alpha = alpha or r
    layer = (layer_cls or LoRALinear)(W0, r, alpha)
    hist = []
    # TODO: loop: compute E, grads for A and B, update both, record loss
    raise NotImplementedError

In [ ]:
# @title Tests for Task 3: run after implementing
def _test_train_lora(f):
    rng = np.random.default_rng(3); d_in, d_out, N = 32, 24, 400
    W0 = rng.normal(0, 1 / np.sqrt(d_in), (d_in, d_out))
    Wstar = W0 + rng.normal(size=(d_in, 4)) @ rng.normal(size=(4, d_out)) / d_in
    X = rng.normal(size=(N, d_in)); Y = X @ Wstar
    cls = LoRALinear_solution if f.__name__.endswith('_solution') else LoRALinear
    _, h4 = f(X, Y, W0, 4, layer_cls=cls); _, h1 = f(X, Y, W0, 1, layer_cls=cls)
    assert h4[-1] < 0.02 * h4[0], f"rank 4 should fit the rank-4 update (loss {h4[0]:.4f} -> {h4[-1]:.4f})"
    assert h1[-1] > 0.2 * h4[0], f"rank 1 should NOT be able to fit a rank-4 update (final {h1[-1]:.4f})"

check("train_lora", _test_train_lora, train_lora)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: The LoRA layer
class LoRALinear_solution:
    def __init__(self, W0, r, alpha, seed=0):
        self.W0, self.r, self.alpha = W0, r, alpha
        rng = np.random.default_rng(seed)
        self.A = rng.normal(0, 1 / np.sqrt(W0.shape[0]), (W0.shape[0], r))
        self.B = np.zeros((r, W0.shape[1]))

    def forward(self, x):
        return x @ self.W0 + (self.alpha / self.r) * (x @ self.A) @ self.B

    def merged(self):
        return self.W0 + (self.alpha / self.r) * self.A @ self.B

check("LoRALinear (reference)", _test_LoRALinear, LoRALinear_solution)

In [ ]:
# @title Solution for Task 2: Count trainable parameters
def lora_param_count_solution(targets, n_layers, r):
    return n_layers * sum(r * (a + b) for a, b in targets)

check("lora_param_count (reference)", _test_lora_param_count, lora_param_count_solution)

In [ ]:
# @title Solution for Task 3: Train only A and B
def train_lora_solution(X, Y, W0, r, alpha=None, steps=2000, lr=0.05, layer_cls=None):
    alpha = alpha or r
    L = (layer_cls or LoRALinear_solution)(W0, r, alpha); s = alpha / r; N = len(X); hist = []
    for _ in range(steps):
        XA = X @ L.A
        E = X @ W0 + s * XA @ L.B - Y
        hist.append((E ** 2).sum() / N)
        gB = 2 * s / N * XA.T @ E
        gA = 2 * s / N * X.T @ (E @ L.B.T)
        L.A -= lr * gA; L.B -= lr * gB
    return L, hist

check("train_lora (reference)", _test_train_lora, train_lora_solution)

---
## Experiments
**Rank sweep.** Fit the same rank-4 update with r = 1…8. The error drops sharply once r reaches the true rank, which is the LoRA bet: fine-tuning updates have low "intrinsic rank". (The paper finds even r = 1–4 works for $W_q$ and $W_v$ on GPT-3.) Then the memory arithmetic: full fine-tuning with Adam keeps gradients and two optimizer moments for **every** parameter; LoRA keeps them only for A and B.

In [ ]:
rng = np.random.default_rng(3); d_in, d_out, N = 32, 24, 400
W0 = rng.normal(0, 1 / np.sqrt(d_in), (d_in, d_out)); Wstar = W0 + rng.normal(size=(d_in, 4)) @ rng.normal(size=(4, d_out)) / d_in
X = rng.normal(size=(N, d_in)); Y = X @ Wstar
for r in [1, 2, 3, 4, 6, 8]:
    _, h = train_lora_solution(X, Y, W0, r)
    print(f"r={r}: final loss {h[-1]:.5f}  (trainable {lora_param_count_solution([(d_in, d_out)], 1, r)} vs full {d_in * d_out})")
P = 8.03e9; lora = lora_param_count_solution([(4096, 4096), (4096, 1024)], 32, 8)
print(f"\nLlama 3 8B, BF16 weights + FP32 grads + 2 FP32 Adam moments (rough):")
print(f"  full fine-tune: {(2 * P + 12 * P) / 2**30:,.0f} GiB   LoRA r=8 on Wq,Wv: {(2 * P + 12 * lora) / 2**30:,.1f} GiB (frozen weights dominate)")

---
## Takeaways
* LoRA freezes $W_0$ and learns $\Delta W = AB$ of rank r. B starts at zero, so training starts exactly at the pretrained model.
* After training, merge $W_0 + \frac{\alpha}{r}AB$ into one matrix: no extra inference latency, unlike adapter layers.
* Trainable parameters shrink by orders of magnitude (the paper: 10,000× fewer for GPT-3 175B, 3× less GPU memory), and you can keep many small adapters for one base model.